In [ ]:
import json
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ==================== 1. 参数配置 ====================

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "outputs" / "bert_three_class"

TRAIN_FILES = [
    DATA_DIR / "yq.txt",
    DATA_DIR / "自己标注-三分类.txt"
]

PREDICT_FILE = DATA_DIR / "remaining_data.csv"

PRETRAINED_MODEL = "bert-base-chinese"

NUM_LABELS = 3
MAX_LENGTH = 128
RANDOM_SEED = 42

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
PREDICT_BATCH_SIZE = 32

EPOCHS = 3
LEARNING_RATE = 2e-5

TXT_SEPARATOR = " "

LABEL_NAMES = {
    0: "Negative",
    1: "Positive",
    2: "Neutral"
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(RANDOM_SEED)


# ==================== 2. 读取和整合数据 ====================

def load_training_data(file_path):
    data = pd.read_csv(
        file_path,
        sep=TXT_SEPARATOR,
        encoding="utf-8"
    )

    required_columns = {"review", "label"}
    if not required_columns.issubset(data.columns):
        raise ValueError(
            f"{file_path.name} 必须包含 review 和 label 两列"
        )

    return data[["review", "label"]]


frames = []

for file_path in TRAIN_FILES:
    if not file_path.exists():
        raise FileNotFoundError(f"找不到训练文件：{file_path}")

    frames.append(load_training_data(file_path))

df = pd.concat(frames, ignore_index=True)

df["review"] = df["review"].fillna("").astype(str).str.strip()
df["label"] = pd.to_numeric(df["label"], errors="coerce")

before_cleaning = len(df)

df = df.dropna(subset=["label"])
df = df[df["review"] != ""].copy()
df["label"] = df["label"].astype(int)

if not set(df["label"].unique()).issubset({0, 1, 2}):
    raise ValueError("发现非 0、1、2 的情感标签，请检查原始数据")

if set(df["label"].unique()) != {0, 1, 2}:
    raise ValueError("训练数据必须包含 0、1、2 三个类别")

# 检查同一文本是否存在不同标签
conflicts = df.groupby("review")["label"].nunique()
conflicts = conflicts[conflicts > 1]

if not conflicts.empty:
    raise ValueError(
        f"发现 {len(conflicts)} 条文本存在不同情感标签，请人工核查"
    )

# 删除完全重复的文本，避免重复文本跨训练集和验证集
duplicate_count = df["review"].duplicated().sum()
df = df.drop_duplicates(subset="review").reset_index(drop=True)

print("数据整合完成")
print("原始数据量：", before_cleaning)
print("清理无效记录及重复文本后：", len(df))
print("删除的重复文本数量：", duplicate_count)

print("\n训练数据情感标签分布：")
print(df["label"].value_counts().sort_index())


# ==================== 3. 划分训练集和验证集 ====================

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df["label"]
)

print("\n训练集数量：", len(train_df))
print("验证集数量：", len(valid_df))


# ==================== 4. 加载 BERT 模型 ====================

tokenizer = BertTokenizerFast.from_pretrained(PRETRAINED_MODEL)

model = BertForSequenceClassification.from_pretrained(
    PRETRAINED_MODEL,
    num_labels=NUM_LABELS,
    id2label=LABEL_NAMES,
    label2id={v: k for k, v in LABEL_NAMES.items()}
)


# ==================== 5. 数据分词 ====================

def tokenize_function(batch):
    return tokenizer(
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_ds = Dataset.from_pandas(
    train_df[["review", "label"]].reset_index(drop=True),
    preserve_index=False
)

valid_ds = Dataset.from_pandas(
    valid_df[["review", "label"]].reset_index(drop=True),
    preserve_index=False
)

train_ds = train_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=["review"]
)

valid_ds = valid_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=["review"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# ==================== 6. 模型评价指标 ====================

def compute_metrics(pred):
    logits = pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)
    labels = pred.label_ids

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": precision_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            labels, predictions, average="weighted", zero_division=0
        )
    }


# ==================== 7. 训练参数 ====================

training_params = {
    "output_dir": str(OUTPUT_DIR / "checkpoints"),
    "num_train_epochs": EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "logging_dir": str(OUTPUT_DIR / "logs"),
    "logging_steps": 100,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy",
    "greater_is_better": True,
    "save_total_limit": 2,
    "seed": RANDOM_SEED,
    "data_seed": RANDOM_SEED,
    "report_to": "none"
}

# 兼容不同 Transformers 版本的参数名称
if "eval_strategy" in inspect.signature(
    TrainingArguments.__init__
).parameters:
    training_params["eval_strategy"] = "epoch"
else:
    training_params["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_params)


# ==================== 8. 模型训练 ====================

trainer_params = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": valid_ds,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics
}

# 兼容不同 Transformers 版本的分词器参数
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_params["processing_class"] = tokenizer
else:
    trainer_params["tokenizer"] = tokenizer

trainer = Trainer(**trainer_params)

print("\n开始训练 BERT 模型")
trainer.train()


# ==================== 9. 保存模型 ====================

MODEL_DIR = OUTPUT_DIR / "model"
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print("\n模型已保存：", MODEL_DIR)


# ==================== 10. 验证集评价 ====================

eval_results = trainer.evaluate()

pred_output = trainer.predict(valid_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print("\n模型综合评价：")
for key, value in eval_results.items():
    print(f"{key}: {value}")

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Positive", "Neutral"],
    output_dict=True,
    zero_division=0
)

print("\n各类别评价：")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Positive", "Neutral"],
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("\n混淆矩阵：")
print(cm)

# 保存评价结果
with open(OUTPUT_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        eval_results,
        f,
        ensure_ascii=False,
        indent=4,
        default=float
    )

pd.DataFrame(report).T.to_csv(
    OUTPUT_DIR / "classification_report.csv",
    encoding="utf-8-sig"
)

pd.DataFrame(
    cm,
    index=["True_Negative", "True_Positive", "True_Neutral"],
    columns=["Pred_Negative", "Pred_Positive", "Pred_Neutral"]
).to_csv(
    OUTPUT_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig"
)


# ==================== 11. 全量文本情感预测 ====================

def predict_texts(texts, model, tokenizer):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.to(device)
    model.eval()

    predictions = []

    for start in range(0, len(texts), PREDICT_BATCH_SIZE):
        batch_texts = texts[start:start + PREDICT_BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        batch_predictions = logits.argmax(dim=-1).cpu().tolist()
        predictions.extend(batch_predictions)

    return predictions


if PREDICT_FILE.exists():

    inference_df = pd.read_csv(
        PREDICT_FILE,
        encoding="utf-8-sig"
    )

    if "text" in inference_df.columns:
        text_column = "text"
    elif len(inference_df.columns) > 6:
        text_column = inference_df.columns[6]
        print("\n未找到 text 列，使用第 7 列：", text_column)
    else:
        raise ValueError("预测文件中没有找到文本列")

    texts = (
        inference_df[text_column]
        .fillna("")
        .astype(str)
        .tolist()
    )

    inference_df["predicted_label"] = predict_texts(
        texts,
        trainer.model,
        tokenizer
    )

    inference_df["predicted_sentiment"] = (
        inference_df["predicted_label"].map(LABEL_NAMES)
    )

    output_file = OUTPUT_DIR / "sentiment_predictions.csv"

    inference_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n情感预测完成")
    print("预测数据量：", len(inference_df))
    print("预测结果：", output_file)

else:
    print("\n未找到全量预测文件，已完成模型训练和验证。")


print("\n全部任务完成。")